# Integridade dos Dados: UCI vs. Bezdek vs. Scikit-Learn

Este notebook documenta as inconsistências históricas do dataset Iris.

O objetivo é demonstrar, com código reproduzível:
1. Quais amostras diferem entre a versão original do UCI (`iris.data`) e a versão corrigida por Bezdek et al. (1999) (`bezdekIris.data`);
2. Que a versão carregada pelo Scikit-Learn (`load_iris()`) é idêntica à versão corrigida.

> **Referência:** BEZDEK, J. C. et al. *Will the real iris data please stand up?*  
> IEEE Transactions on Fuzzy Systems, v. 7, n. 3, p. 368–369, 1999.  
> DOI: 10.1109/91.771092

In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris

# Nomes das colunas conforme o arquivo iris.names do UCI
COLUNAS = [
    'sepal length (cm)',
    'sepal width (cm)',
    'petal length (cm)',
    'petal width (cm)',
    'class'
]

# Carregar os dois arquivos do repositório UCI
df_uci     = pd.read_csv('uci-dataset-iris/iris.data',      header=None, names=COLUNAS)
df_bezdek  = pd.read_csv('uci-dataset-iris/bezdekIris.data', header=None, names=COLUNAS)

print(f'UCI original : {len(df_uci)} amostras')
print(f'Bezdek (corrigido): {len(df_bezdek)} amostras')

UCI original : 150 amostras
Bezdek (corrigido): 150 amostras


## 1. Encontrando as diferenças entre os dois arquivos

In [3]:
# Comparar apenas as colunas numéricas (a coluna 'class' é idêntica nos dois arquivos)
colunas_numericas = COLUNAS[:-1]

# Identificar linhas onde há pelo menos um valor diferente
mascara_diff = ~(df_uci[colunas_numericas] == df_bezdek[colunas_numericas]).all(axis=1)
indices_diff = df_uci.index[mascara_diff].tolist()

print(f'Número de amostras divergentes: {len(indices_diff)}')
print(f'Índices (base 0): {indices_diff}')

Número de amostras divergentes: 2
Índices (base 0): [34, 37]


## 2. Inspecionando as amostras divergentes

In [4]:
for idx in indices_diff:
    print(f'\n=== Amostra índice {idx} (linha {idx + 1} do arquivo) ===')
    
    comparacao = pd.DataFrame({
        'UCI original (iris.data)':      df_uci.loc[idx, colunas_numericas],
        'Bezdek corrigido (bezdekIris.data)': df_bezdek.loc[idx, colunas_numericas],
    })
    
    # Destacar apenas os campos que diferem
    diffs = comparacao[comparacao['UCI original (iris.data)'] != comparacao['Bezdek corrigido (bezdekIris.data)']]
    
    print(comparacao.to_string())
    print('\nCampos alterados:')
    print(diffs.to_string())


=== Amostra índice 34 (linha 35 do arquivo) ===
                   UCI original (iris.data)  Bezdek corrigido (bezdekIris.data)
sepal length (cm)                       4.9                                 4.9
sepal width (cm)                        3.1                                 3.1
petal length (cm)                       1.5                                 1.5
petal width (cm)                        0.1                                 0.2

Campos alterados:
                  UCI original (iris.data)  Bezdek corrigido (bezdekIris.data)
petal width (cm)                       0.1                                 0.2

=== Amostra índice 37 (linha 38 do arquivo) ===
                   UCI original (iris.data)  Bezdek corrigido (bezdekIris.data)
sepal length (cm)                       4.9                                 4.9
sepal width (cm)                        3.1                                 3.6
petal length (cm)                       1.5                                 1.4
petal

## 3. Confirmando que o Scikit-Learn usa a versão corrigida

In [5]:
# Carregar o dataset via sklearn
iris_sklearn = load_iris()
df_sklearn = pd.DataFrame(data=iris_sklearn.data, columns=colunas_numericas)

# Comparar sklearn com bezdekIris.data (devem ser idênticos)
diff_sklearn_bezdek = ~(df_sklearn == df_bezdek[colunas_numericas]).all(axis=1)
n_diff_sklearn_bezdek = diff_sklearn_bezdek.sum()

# Comparar sklearn com iris.data (deve haver diferenças)
diff_sklearn_uci = ~(df_sklearn == df_uci[colunas_numericas]).all(axis=1)
n_diff_sklearn_uci = diff_sklearn_uci.sum()

print('--- sklearn vs. bezdekIris.data (versão corrigida) ---')
print(f'Amostras diferentes: {n_diff_sklearn_bezdek}')
print('✓ Idênticos.' if n_diff_sklearn_bezdek == 0 else '✗ Divergem!')

print('\n--- sklearn vs. iris.data (versão UCI original) ---')
print(f'Amostras diferentes: {n_diff_sklearn_uci}')
print('✓ Idênticos.' if n_diff_sklearn_uci == 0 else f'✗ Divergem em {n_diff_sklearn_uci} amostra(s).')

--- sklearn vs. bezdekIris.data (versão corrigida) ---
Amostras diferentes: 0
✓ Idênticos.

--- sklearn vs. iris.data (versão UCI original) ---
Amostras diferentes: 2
✗ Divergem em 2 amostra(s).


## 4. Resumo

A célula abaixo consolida os resultados em um único quadro de referência.

In [6]:
print('RESUMO DAS INCONSISTÊNCIAS\n')
print(f'Total de amostras no dataset : 150')
print(f'Amostras com divergência     : {len(indices_diff)} (índices {indices_diff})')
print(f'Espécie afetada              : Iris-setosa (primeiras 50 amostras)')
print()

for idx in indices_diff:
    row_uci = df_uci.loc[idx, colunas_numericas]
    row_bzd = df_bezdek.loc[idx, colunas_numericas]
    campos_alterados = [c for c in colunas_numericas if row_uci[c] != row_bzd[c]]
    
    print(f'Índice {idx}:')
    for campo in campos_alterados:
        print(f'  {campo}: {row_uci[campo]} (UCI) → {row_bzd[campo]} (Bezdek/sklearn)')

print()
print('Fonte utilizada nos experimentos do artigo: sklearn.datasets.load_iris()')
print('Equivalência com bezdekIris.data confirmada: SIM')

RESUMO DAS INCONSISTÊNCIAS

Total de amostras no dataset : 150
Amostras com divergência     : 2 (índices [34, 37])
Espécie afetada              : Iris-setosa (primeiras 50 amostras)

Índice 34:
  petal width (cm): 0.1 (UCI) → 0.2 (Bezdek/sklearn)
Índice 37:
  sepal width (cm): 3.1 (UCI) → 3.6 (Bezdek/sklearn)
  petal length (cm): 1.5 (UCI) → 1.4 (Bezdek/sklearn)

Fonte utilizada nos experimentos do artigo: sklearn.datasets.load_iris()
Equivalência com bezdekIris.data confirmada: SIM
